---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"

---

## Parameters

In [ ]:
# IMPORTANT PARAMETERS:
year_to_load <- "2018" # Which claims year to load # TODO: maybe add a script that loops through all claims?
split_parts <- 5 # How many parts to split the 12+m row claims file into
rows_to_show <- 10 # How many rows/entries to show in summary tables

# Input:
to_read <- FALSE # TODO: Deprecated and Unused # Whether to forcibly read the whole file again instead of using the split parts created even if available
to_split <- TRUE # TODO: Deprecated, only used when to_sample is TRUE # Whether to split into split_parts parts (i.e. to fit in 32gb RAM). 
to_sample <- TRUE # Whether to sample each split_parts part by sample_size_divisor (useful when iterating through code runs in quick succession)
sample_size_divisor <- 125 # Sample size divisor: Formula for sample size is total_rows / split_parts / sample_size_divisor

# Output:
to_write <- TRUE # Whether to write out intermediate files and caches (i.e. part files, sample files). TODO: upload to BQ as well
to_group <- TRUE # Whether to export for the batch grouper or not

# Debug:
to_profvis <- FALSE # Conduct runtime duration analysis via profvis or not
to_view_checks <- TRUE # Whether to view checks and print statements
to_view_checks_parallelized <- FALSE # Whether to view intermediate per part/chunk checks and print statements (not consolidated) when parallelized
to_parallelize <- TRUE # Whether to parallelize each split_parts part into availableCores() - 1 chunks. Cuts down processing time from 120min to 15min.
intermediate_rows_to_show <- Inf # Per part/chunk rows_to_show (leave at Inf)

drop_cols <- c( # Which columns to drop
  paste0("ICDCODE", 13:14), # Start
  "ICCODED15", # note that ICDCODE15 is misspelled as ICCODED15 in all claims
  paste0("ICDCODE", 16:170) # Continuation
)

seed <- 123 # Seed for reproducibility (Important for stuff like randomly choosing a pdx among multiple possible options)
set.seed(seed) # Setting the seed
global_seed <- seed # global_seed for future_lapply parts for parallelized operations

options(future.globals.maxSize = 1024 * 1024^2) # Allowing each future_lapply session to use more memory


## Load Required Libraries & Initial Functions

In [ ]:
options(verbose = FALSE) # Hide verbose output for script and library loading
options(warn = -1) # Hide warnings for script sourcing and library loading
library(here) # Library here() so scipts can be loaded


In [ ]:
scripts_to_source <- c( # List of scripts to source
  "00_libraries.R",
  "01_data-formats.R",
  "02_file-paths.R",
  "03_general-functions.R",
  "04a_clean-data-functions.R",
  "04b_chunk-functions.R",
  "04c_part-functions.R",
  "05_io-functions.R",
  "06_icd-functions.R",
  "07_rvs-functions.R",
  "08_pdx-functions.R",
  "09_grouper-functions.R",
  "10_timing-functions.R",
  "11_debug-functions.R",
  "12_summary-functions.R"
)

for (script in scripts_to_source) { # Loop to source above scripts
  source(here("data-cleaning", "r_scripts", script))
}

tic("Total execution time:") # Start total execution timer


In [ ]:
# TODO: figure out a way to return to default outputs
# since verbose = TRUE is way too verbose compared to default
options(warn = 1) # Reenable warnings; see above comments


## Load Mapping Data

In [ ]:
# Read in all rvs codes and turn to character for further processing
proc <- fread(here(path_to_excel, "proc.csv"))
proc[, CODE := as.character(CODE)]

# Read in icd9cm equivalents of rvs codes
rvs_icd9 <- fread(here(path_to_aux, "rvs_icd9cm.csv"),
  select = c("rvs", "icd9cm")
)

# Convert to character for further processing
rvs_icd9[, rvs := as.character(rvs)]

# Convert to character and also remove decimals
# whilst keeping trailing zeroes
rvs_icd9[, icd9cm := as.character(icd9cm * 100)]

# Merge with proc from above, to be able to classify by DRGUSE
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)],
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
)

# Filter by DRGUSE
rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE]

# Remove DRGUSE and filter out NAs
rvs_icd9 <- rvs_icd9[!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]

# Read in PHIC all case rates
acr_rvs <- fread(here(path_to_aux, "acr_rvs.csv"))

# Read in the thai icd10 library
tdrg_icd10 <- fread(here(path_to_aux, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")

# Subset and assign the result to acc_pdx
acc_pdx <- unique(tdrg_icd10[ACCPDX == "Y", CODE])


## Read, Process, Export Data (Looping through all parts)

In [ ]:
all_parts_summaries <- list()
processing_times <- numeric(split_parts)
num_cores <- availableCores() - 1

if (to_profvis) {
  p <- profvis({
    # Start main execution logic
    split_and_save_parts() # Read, split, and save partial files

    if (to_parallelize) {
      # Start the parallelization session
      plan(multisession, workers = num_cores)
    } else {
      # Status quo; i.e. remain sequential
      plan(sequential)
    }

    # For each partial file in N (split_parts) files,
    for (part in 1:split_parts) {
      # Process the partial file with or without parallelization
      result <- process_part(
        part, num_cores, to_view_checks, global_seed,
        intermediate_rows_to_show, rvs_icd9, tdrg_icd10,
        acc_pdx, to_parallelize, to_write, to_group, to_sample
      )

      # Save partial summaries to a list
      all_parts_summaries[[part]] <- result$combined_summary

      # Save partial processing time to a list
      processing_times[part] <- result$processing_time

      # Print status update and ETA
      # VS Code: Updates are shown after complete execution
      # Positron: Updates are shown live
      print_status_update(part, split_parts, processing_times)

      # Save resulting partial dt to variable dt, to be used for
      # Speed calculations
      dt <- result$dt
    }
    # End the parallelization session
    plan(sequential)

    # End main execution logic
  })
  # Save profvis as html
  htmlwidgets::saveWidget(p,
    file = here("git-ignored-files", "profvis", "profvis.html")
  )
} else {
  # Main execution logic
  stop("You forgot to include code in the not to profvis section")
}

# Summaries are consolidated from 5 split_parts * 15 chunks = 75 sub outputs
print_summary_tables( # Print final summaries
  combine_parts_summaries( # input
    all_parts_summaries, intermediate_rows_to_show # input cont'd.
  ), rows_to_show # param
)


## Runtime Estimation

### Stop Timer

In [ ]:
# Stop the timer and capture total time
toc_data <- toc(log = TRUE)
# Compute total time
total_time <- toc_data$toc - toc_data$tic


### Calculate Speed

In [ ]:
# Calculate how many rows were actually processed,
# Not just how many rows exist in the actual dataframe
# TODO: rename total_rows to something else to avoid confusion
if (to_sample) {
  total_rows <- nrow(dt) * split_parts * sample_size_divisor
} else {
  total_rows <- nrow(dt) * split_parts
}


In [ ]:
# Print speed computation using above cell's data
print_time_estimates(dt, total_time, total_rows)


## Debugging

In [ ]:
# Consolidate all r_scripts scripts into everything.R
# Useful for debugging

# in case we want to run this cell independently:
library(here)
source(here("data-cleaning", "r_scripts", "11_debug-functions.R"))

concatenate_r_files(
  here("data-cleaning", "r_scripts"), # input
  paste0(here("data-cleaning", "everything", "everything.R")) # output
)


To extract all code portions of this ipynb file (run in VS Code terminal):

jupyter nbconvert --no-prompt --to script data-cleaning/drg-cleaning.ipynb --output everything/drg-cleaning

